# Improved ScienceQA Solution — LoRA + Letter Log-Prob Scoring

**Key fixes over previous attempts:**
- **LR: 2e-5** (was 5e-4 → 25x too high, destroyed pretrained knowledge)
- **IMG_SIZE: 384** (SmolVLM native, was 160 in LoRA notebook)
- **LoRA r=4, q_proj+v_proj** → ~276k params (TA target: ~300k)
- **1 epoch** (TA hint #3)
- **Cosine LR schedule** with warmup + gradient clipping
- **Letter log-prob scoring** at inference (best zero-shot method at 0.635)
- **Batch size 16** for inference (TA hint #11: 20-30x speedup)

**Workflow:**
1. Quick loss check on 10% (verify setup works)
2. Train full dataset, 1 epoch
3. Evaluate on validation
4. Generate submission.csv

In [1]:
# ── 0. Install ──
!pip install -q kagglehub
!pip install -q "transformers==4.57.1" "peft==0.18.0" bitsandbytes accelerate "pandas==2.2.2" "pillow==11.3.0" tqdm

In [2]:
# ── 1. Download Data ──
import kagglehub
from pathlib import Path

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
COMPETITION_NAME = "pixels-to-predictions"
COMP_DIR = Path(kagglehub.competition_download(COMPETITION_NAME))
print("Downloaded to:", COMP_DIR)

Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions


In [4]:
# ── 2. Imports & Config ──
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json, re, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVision2Seq, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
IMG_SIZE = 224    # NOT 384! Large images cause text truncation
CHOICE_LETTERS = "ABCDEFGHIJ"

# Training
BATCH_SIZE_TRAIN = 1
GRAD_ACCUM_STEPS = 16
LR = 2e-5              # KEY: was 5e-4 before (way too high)
EPOCHS = 1              # TA: 1 epoch → baseline
WARMUP_RATIO = 0.05

# LoRA (~276k params)
LORA_R = 4
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "v_proj"]

# Inference
BATCH_SIZE_INFER = 16

SAVE_DIR = Path("/content/lora_improved_v1")

print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  {torch.cuda.get_device_name(0)}")

GPU: True
  Tesla T4


In [5]:
# ── 3. Load Data ──
DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists():
    DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Auto-detect image root
example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
DATA_ROOT = matches[0].parents[2]
print("DATA_ROOT:", DATA_ROOT)

Train: 3109 | Val: 1048 | Test: 1008
DATA_ROOT: /root/.cache/kagglehub/competitions/pixels-to-predictions/images


In [6]:
# ── 4. Prompt + Image Helpers ──
def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def build_prompt(row, include_answer=False):
    choices = row["choices"]
    choices_text = "\n".join(
        f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    prompt = "<image>\n"
    prompt += "You are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY ONE LETTER (A, B, C, etc). No explanation.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint:    prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"
    if include_answer:
        prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"
    return prompt

def load_image(row):
    path = DATA_ROOT / row["image_path"]
    return Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)

print(build_prompt(train_df.iloc[0], include_answer=True)[:200])

<image>
You are solving a science multiple-choice question.
Use the image and text to choose the best answer.
Answer with ONLY ONE LETTER (A, B, C, etc). No explanation.

Lecture: Animals increase the


In [7]:
# ── 5. Load Model + Apply LoRA ──
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} (limit: 5,000,000)")
assert trainable <= 5_000_000

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


trainable params: 557,056 || all params: 508,039,360 || trainable%: 0.1096
Trainable: 557,056 (limit: 5,000,000)


In [8]:
# ── DEBUG: Check what build_prompt actually produces ──
for idx in range(4):
    row = train_df.iloc[idx]
    prompt_no_answer = build_prompt(row, include_answer=False)
    prompt_with_answer = build_prompt(row, include_answer=True)

    print(f"=== Sample {idx} (answer={row['answer']}, letter={CHOICE_LETTERS[int(row['answer'])]}) ===")
    print(f"Prompt ends with: ...'{prompt_no_answer[-30:]}'")
    print(f"Full ends with:   ...'{prompt_with_answer[-30:]}'")

    # The difference should be just " X" (the answer letter)
    if prompt_with_answer.startswith(prompt_no_answer):
        diff = prompt_with_answer[len(prompt_no_answer):]
        print(f"Difference: '{diff}'")
    else:
        print("⚠️ WARNING: full text does NOT start with prompt text!")
        # Find where they diverge
        for j in range(min(len(prompt_no_answer), len(prompt_with_answer))):
            if prompt_no_answer[j] != prompt_with_answer[j]:
                print(f"  Diverge at char {j}:")
                print(f"  Prompt: ...'{prompt_no_answer[max(0,j-20):j+20]}'")
                print(f"  Full:   ...'{prompt_with_answer[max(0,j-20):j+20]}'")
                break
    print()

=== Sample 0 (answer=2, letter=C) ===
Prompt ends with: ...'ll become adult frogs

Answer:'
Full ends with:   ...' become adult frogs

Answer: C'
Difference: ' C'

=== Sample 1 (answer=0, letter=A) ===
Prompt ends with: ...' around other females

Answer:'
Full ends with:   ...'round other females

Answer: A'
Difference: ' A'

=== Sample 2 (answer=1, letter=B) ===
Prompt ends with: ...'bs of other lionesses

Answer:'
Full ends with:   ...' of other lionesses

Answer: B'
Difference: ' B'

=== Sample 3 (answer=1, letter=B) ===
Prompt ends with: ...'pring at a given time

Answer:'
Full ends with:   ...'ing at a given time

Answer: B'
Difference: ' B'



In [9]:
# ── DEBUG: Check truncation behavior ──
from PIL import Image
import torch

# Build a long prompt from a sample with lecture+hint
for idx in range(8):
    row = train_df.iloc[idx]
    full_text = build_prompt(row, include_answer=True)
    img = load_image(row)

    # Default processor call
    inputs = processor(text=[full_text], images=[img], return_tensors="pt")
    seq_len = inputs["input_ids"].shape[1]
    last_tok = processor.tokenizer.decode([inputs["input_ids"][0, -1].item()]).strip()
    expected = CHOICE_LETTERS[int(row["answer"])]

    # No truncation
    inputs_nt = processor(text=[full_text], images=[img], return_tensors="pt", truncation=False)
    seq_len_nt = inputs_nt["input_ids"].shape[1]
    last_tok_nt = processor.tokenizer.decode([inputs_nt["input_ids"][0, -1].item()]).strip()

    status = "✅" if last_tok == expected else "❌"
    print(f"Sample {idx}: default={seq_len} tokens (last='{last_tok}'), "
          f"no_trunc={seq_len_nt} tokens (last='{last_tok_nt}'), expected='{expected}' {status}")

print(f"\nTokenizer max_length: {processor.tokenizer.model_max_length}")

Sample 0: default=1677 tokens (last='C'), no_trunc=1677 tokens (last='C'), expected='C' ✅
Sample 1: default=1651 tokens (last='A'), no_trunc=1651 tokens (last='A'), expected='A' ✅
Sample 2: default=1650 tokens (last='B'), no_trunc=1650 tokens (last='B'), expected='B' ✅
Sample 3: default=1655 tokens (last='B'), no_trunc=1655 tokens (last='B'), expected='B' ✅
Sample 4: default=1652 tokens (last='B'), no_trunc=1652 tokens (last='B'), expected='B' ✅
Sample 5: default=1615 tokens (last='C'), no_trunc=1615 tokens (last='C'), expected='C' ✅
Sample 6: default=1602 tokens (last='C'), no_trunc=1602 tokens (last='C'), expected='C' ✅
Sample 7: default=1629 tokens (last='D'), no_trunc=1629 tokens (last='D'), expected='D' ✅

Tokenizer max_length: 8192


In [11]:
# ── 6. Dataset & Collator (memory-safe) ──
import io, pickle

# Pre-process images to tensors and save to disk (not RAM)
CACHE_DIR = Path("/content/img_cache")
CACHE_DIR.mkdir(exist_ok=True)

class VQATrainDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        answer_letter = CHOICE_LETTERS[int(row["answer"])]
        image = load_image(row)
        full_text = build_prompt(row, include_answer=True)

        # Process individually (correct tokenization)
        inputs = processor(text=[full_text], images=[image], return_tensors="pt", truncation=False)

        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "answer_letter": answer_letter,
        }

def collate_train(batch):
    max_len = max(x["input_ids"].shape[0] for x in batch)
    pad_id = processor.tokenizer.pad_token_id

    input_ids_list, attention_mask_list, labels_list, pixel_values_list = [], [], [], []

    for x in batch:
        seq_len = x["input_ids"].shape[0]
        pad_len = max_len - seq_len

        input_ids = torch.cat([
            torch.full((pad_len,), pad_id, dtype=x["input_ids"].dtype),
            x["input_ids"],
        ])
        attention_mask = torch.cat([
            torch.zeros(pad_len, dtype=x["attention_mask"].dtype),
            x["attention_mask"],
        ])

        labels = torch.full_like(input_ids, -100)
        labels[-1] = input_ids[-1]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)
        pixel_values_list.append(x["pixel_values"])

    return {
        "input_ids": torch.stack(input_ids_list),
        "attention_mask": torch.stack(attention_mask_list),
        "labels": torch.stack(labels_list),
        "pixel_values": torch.stack(pixel_values_list),
    }

# Quick verify (just 4 samples, no RAM issue)
_test_ds = VQATrainDataset(train_df.head(4))
_test_batch = collate_train([_test_ds[j] for j in range(4)])
for i in range(4):
    last_label = _test_batch["labels"][i][_test_batch["labels"][i] != -100]
    decoded = processor.tokenizer.decode(last_label).strip()
    print(f"  Sample {i}: label='{decoded}' ✅" if len(decoded)==1 else f"  Sample {i}: label='{decoded}' ❌")

  Sample 0: label='C' ✅
  Sample 1: label='A' ✅
  Sample 2: label='B' ✅
  Sample 3: label='B' ✅


In [12]:
# ── 7. Loss Trend Check ──
subset = train_df.sample(frac=0.10, random_state=SEED).reset_index(drop=True)
ds_check = VQATrainDataset(subset)
loader_check = DataLoader(ds_check, batch_size=BATCH_SIZE_TRAIN, shuffle=True,
                           collate_fn=collate_train, num_workers=0)

model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01
)
losses = []
optimizer.zero_grad()

N_CHECK_STEPS = 200
for step, batch in enumerate(loader_check):
    if step >= N_CHECK_STEPS: break
    batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
    out = model(**batch)
    loss = out.loss / GRAD_ACCUM_STEPS
    loss.backward()
    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        optimizer.step(); optimizer.zero_grad(set_to_none=True)
    losses.append(out.loss.item())
    if (step+1) % 32 == 0:
        print(f"Step {step+1}: loss={losses[-1]:.4f} avg32={np.mean(losses[-32:]):.4f}")

w = 32
print(f"\nFirst {w} avg: {np.mean(losses[:w]):.4f}")
print(f"Last {w} avg:  {np.mean(losses[-w:]):.4f}")

model.eval(); model.config.use_cache = True
del optimizer, loader_check, ds_check
gc.collect(); torch.cuda.empty_cache()

Step 32: loss=2.5029 avg32=0.7258
Step 64: loss=0.1028 avg32=1.0610
Step 96: loss=0.0458 avg32=0.7620
Step 128: loss=1.6697 avg32=0.6713
Step 160: loss=0.8877 avg32=1.0089
Step 192: loss=1.2758 avg32=0.5582

First 32 avg: 0.7258
Last 32 avg:  0.2995


In [13]:
# ── 8. Full Training (1 epoch, full dataset) ──
# Reload clean model + LoRA from scratch
del model; gc.collect(); torch.cuda.empty_cache()

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model = get_peft_model(model, lora_config)

# ── FIX: proper setup order for PEFT + gradient checkpointing ──
model.train()
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

ds_full = VQATrainDataset(train_df)
loader_full = DataLoader(ds_full, batch_size=BATCH_SIZE_TRAIN, shuffle=True,
                          collate_fn=collate_train, num_workers=2, pin_memory=True)

total_steps = len(loader_full)
opt_steps = total_steps // GRAD_ACCUM_STEPS

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01
)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=max(1, int(opt_steps * WARMUP_RATIO)),
    num_training_steps=opt_steps,
)

loss_history = []
optimizer.zero_grad()
pbar = tqdm(loader_full, desc="Training (1 epoch)")

for step, batch in enumerate(pbar):
    batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
    out = model(**batch)
    loss = out.loss / GRAD_ACCUM_STEPS
    loss.backward()

    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    loss_history.append(out.loss.item())
    if (step+1) % 100 == 0:
        pbar.set_postfix(loss=f"{loss_history[-1]:.4f}",
                         avg100=f"{np.mean(loss_history[-100:]):.4f}",
                         lr=f"{scheduler.get_last_lr()[0]:.2e}")

model.eval(); model.config.use_cache = True
torch.cuda.empty_cache()
print(f"\nDone! First 50 avg: {np.mean(loss_history[:50]):.4f}, Last 50 avg: {np.mean(loss_history[-50:]):.4f}")

Training (1 epoch):   0%|          | 0/3109 [00:00<?, ?it/s]


Done! First 50 avg: 0.7753, Last 50 avg: 0.5321


In [14]:
# ── 9. Save Adapter ──
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)
!ls -la {SAVE_DIR}

Saved to: /content/lora_improved_v1
total 6984
drwxr-xr-x 2 root root    4096 Apr 26 02:20 .
drwxr-xr-x 1 root root    4096 Apr 26 02:20 ..
-rw-r--r-- 1 root root     988 Apr 26 02:20 adapter_config.json
-rw-r--r-- 1 root root 2253488 Apr 26 02:20 adapter_model.safetensors
-rw-r--r-- 1 root root    4739 Apr 26 02:20 added_tokens.json
-rw-r--r-- 1 root root     403 Apr 26 02:20 chat_template.jinja
-rw-r--r-- 1 root root  466391 Apr 26 02:20 merges.txt
-rw-r--r-- 1 root root     486 Apr 26 02:20 preprocessor_config.json
-rw-r--r-- 1 root root      68 Apr 26 02:20 processor_config.json
-rw-r--r-- 1 root root    5224 Apr 26 02:20 README.md
-rw-r--r-- 1 root root    1077 Apr 26 02:20 special_tokens_map.json
-rw-r--r-- 1 root root   27852 Apr 26 02:20 tokenizer_config.json
-rw-r--r-- 1 root root 3548256 Apr 26 02:20 tokenizer.json
-rw-r--r-- 1 root root  800662 Apr 26 02:20 vocab.json


In [17]:
SAVE_DIR = Path("/content/lora_improved_v1")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved!")

# Also save to Google Drive as backup
from google.colab import drive
drive.mount("/content/drive")
!cp -r /content/lora_improved_v1 /content/drive/MyDrive/

Saved!
Mounted at /content/drive


In [18]:
# ── 10. Letter Log-Prob Scoring Inference ──
def get_candidate_token_ids(tokenizer):
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " "+letter, "\n"+letter, letter+".", " "+letter+"."]
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc: ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids

candidate_ids = get_candidate_token_ids(processor.tokenizer)

def predict_batch(df_batch):
    images = [load_image(row) for _, row in df_batch.iterrows()]
    prompts = [build_prompt(row) for _, row in df_batch.iterrows()]

    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]

    log_probs = torch.log_softmax(logits, dim=-1)
    preds, scores_all = [], []

    for i, (_, row) in enumerate(df_batch.iterrows()):
        scores = []
        for ci in range(len(row["choices"])):
            tids = candidate_ids[CHOICE_LETTERS[ci]]
            scores.append(log_probs[i, tids].max().item())
        preds.append(int(np.argmax(scores)))
        scores_all.append(scores)

    return preds, scores_all

In [20]:
# Clear training memory
model.eval()
model.config.use_cache = True
gc.collect()
torch.cuda.empty_cache()

# Reduce inference batch size
BATCH_SIZE_INFER = 4  # was 16, too big after training

model.eval()
candidate_ids = get_candidate_token_ids(processor.tokenizer)

N_VAL = 200
eval_df = val_df.copy()
eval_df = eval_df.sample(n=min(N_VAL, len(eval_df)), random_state=SEED).reset_index(drop=True)

all_preds, all_scores = [], []
for start in tqdm(range(0, len(eval_df), BATCH_SIZE_INFER), desc="Val"):
    batch = eval_df.iloc[start:start+BATCH_SIZE_INFER]
    p, s = predict_batch(batch)
    all_preds.extend(p); all_scores.extend(s)
    torch.cuda.empty_cache()  # free memory after each batch

eval_df["pred"] = all_preds
eval_df["correct"] = eval_df["pred"] == eval_df["answer"]
acc = eval_df["correct"].mean()
print(f"\nValidation accuracy: {acc:.4f} ({eval_df['correct'].sum()}/{len(eval_df)})")
print(f"Target: > 0.678 (leaderboard baseline)")

Val:   0%|          | 0/50 [00:00<?, ?it/s]


Validation accuracy: 0.6300 (126/200)
Target: > 0.678 (leaderboard baseline)


In [21]:
all_preds = []
for start in tqdm(range(0, len(test_df), BATCH_SIZE_INFER), desc="Test"):
    batch = test_df.iloc[start:start+BATCH_SIZE_INFER]
    p, _ = predict_batch(batch)
    all_preds.extend(p)
    torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": all_preds})
submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv ({len(submission)} rows)")

from google.colab import files
files.download("submission.csv")

Test:   0%|          | 0/252 [00:00<?, ?it/s]

Saved submission.csv (1008 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
# ── 13. (Optional) Zip adapter for Kaggle submission ──
!zip -r lora_improved_v1.zip {SAVE_DIR}
print("Created lora_improved_v1.zip for Kaggle upload")

  adding: content/lora_improved_v1/ (stored 0%)
  adding: content/lora_improved_v1/vocab.json (deflated 59%)
  adding: content/lora_improved_v1/merges.txt (deflated 55%)
  adding: content/lora_improved_v1/special_tokens_map.json (deflated 81%)
  adding: content/lora_improved_v1/README.md (deflated 65%)
  adding: content/lora_improved_v1/tokenizer.json (deflated 82%)
  adding: content/lora_improved_v1/processor_config.json (deflated 12%)
  adding: content/lora_improved_v1/chat_template.jinja (deflated 51%)
  adding: content/lora_improved_v1/tokenizer_config.json (deflated 95%)
  adding: content/lora_improved_v1/adapter_model.safetensors (deflated 7%)
  adding: content/lora_improved_v1/adapter_config.json (deflated 56%)
  adding: content/lora_improved_v1/added_tokens.json (deflated 86%)
  adding: content/lora_improved_v1/preprocessor_config.json (deflated 55%)
Created lora_improved_v1.zip for Kaggle upload


## What to do if accuracy < 0.678

If the LoRA training didn't beat the baseline, try these in order:

1. **Try different LR**: 1e-5, 3e-5, 5e-5 (small range around 2e-5)
2. **Add more LoRA targets**: `["q_proj", "k_proj", "v_proj", "o_proj"]` with r=2 (~276k)
3. **Try larger image size**: SmolVLM supports up to 512
4. **Hard sample mining** (TA hint #7): Run inference on train set, find misclassified samples, train only on those
5. **Prompt variations**: Try without the system instruction, or with "The answer is" instead of "Answer:"
6. **Train on full val+train**: If overfitting isn't an issue (monitor val accuracy during training)